# PolyArc CSG - Demo Notebook

This notebook demonstrates the core functionality of **PolyArc CSG**: converting arbitrary 2D CSG expressions into **exact signed distance fields** by producing intersection-free boundary representations.

## The Problem

Standard CSG evaluation using `min`/`max` pooling produces **inexact SDFs** for complex Boolean expressions. This causes:
- Incorrect gradients away from zero-level sets
- Distorted offset curves
- Unreliable downstream operations

## The Solution

PolyArc CSG resolves all boundary intersections through DNF/CNF transformations, producing a **valid PolySet** where no curves intersect. This guarantees exact SDF evaluation.

---

## Setup

First, we import the necessary libraries and create a Sketcher for coordinate transformations:


In [ ]:
import geolipi.symbolic as gls
import numpy as np
from migumi.utils.vis import draw_contour_plot, fig_to_image
from geolipi.torch_compute import recursive_evaluate, Sketcher
sketcher = Sketcher(resolution=512, n_dims=2)

## Example 1: Complex Union and Difference

This example creates a complex shape by combining:
- A circle and a rotated rectangle (union)
- A circle with a rectangular hole (difference)
- Subtracting another circle from the result

The first plot shows the **standard SDF evaluation** which has inexact distance values at intersection regions:


In [ ]:

expr = gls.Union(
    gls.Circle2D((0.5,)), 
    gls.Translate2D(
        gls.Rectangle2D((0.25, 0.95)), 
        (0.55, 0.0)))
expr = gls.EulerRotate2D(expr, (np.pi/2,))

expr_2 = gls.Translate2D(
    gls.Difference(
        gls.Circle2D((0.25,)), 
        gls.Rectangle2D((0.1, 0.1))),
    (0.50, 0.0))

expr = gls.Union(expr, expr_2)
expr = gls.Difference(
    expr, 
    gls.Translate2D(gls.Circle2D((0.35,)), (-0.25, 0.0)))

output = recursive_evaluate(expr.tensor(), sketcher)
fig = draw_contour_plot(output, 512, add_colorbar=False, levelrate=25, solid_inside=False)
fig.show()

### Resolved PolyArc CSG

Now we convert the expression to a valid PolySet using `expr_to_valid_polyset_expr`. This:
1. Resolves all transforms to explicit PolyArc2D primitives
2. Applies DNF/CNF transformations to flatten the expression
3. Performs geometric Boolean operations to eliminate intersections
4. Returns an equivalent expression with **exact SDF**:


In [ ]:
from polyarc_csg.generic_to_polyset import expr_to_polyarc_expr
from polyarc_csg.valid_polyset import expr_to_valid_polyset_expr
poly_expr = expr_to_valid_polyset_expr(expr.tensor(), sketcher)
output = recursive_evaluate(poly_expr.tensor(), sketcher)
fig = draw_contour_plot(output, 512, add_colorbar=False, levelrate=25, solid_inside=False)

fig.show()


---

## Example 2: Nested Differences with Rectangles

This example demonstrates deeply nested difference operations:
- Concentric circles with alternating subtraction
- A horizontal rectangle added via union
- A vertical rectangle subtracted

Such nested operations are particularly challenging for standard SDF evaluation:


In [ ]:
expr = gls.Difference(
    gls.Circle2D((0.85,)),
    gls.Difference(
        gls.Circle2D((0.65,)),
        gls.Difference(
            gls.Circle2D((0.45,)), gls.Circle2D((0.25,)),
        ),
    )
)
expr = gls.Union(
    expr, gls.Rectangle2D((2.5, 0.20))
)
expr = gls.Difference(expr, gls.Rectangle2D((0.20, 2.5)))
output = recursive_evaluate(expr.tensor(), sketcher)
fig = draw_contour_plot(output, 512, add_colorbar=False, levelrate=25)
fig.show()

### Resolved PolyArc CSG

The resolved expression shows the flattened PolySet representation. Notice how complex nested operations become simple polygon descriptions:


In [ ]:

poly_expr = expr_to_valid_polyset_expr(expr.tensor(), sketcher)
print(poly_expr)

output = recursive_evaluate(poly_expr.tensor(), sketcher)
fig = draw_contour_plot(output, 512, add_colorbar=False, levelrate=25)
fig.show()



---

## Example 3: Translated Copies

This example creates two translated copies of a complex nested circular pattern. This tests the algorithm's ability to handle:
- Multiple disjoint regions
- Interactions between translated copies
- Proper enclosure tree construction


In [ ]:
expr = gls.Difference(
    gls.Circle2D((0.55,)),
    gls.Difference(
        gls.Circle2D((0.35,)),
        gls.Difference(
            gls.Circle2D((0.25,)), gls.Circle2D((0.15,)),
        ),
    )
)

expr = gls.Union(
    gls.Translate2D(expr, (0.0, -0.15)), 
    gls.Translate2D(expr, (0.0, 0.15))
)

# expr = gls.Difference(
#     gls.Rectangle2D((1.5, 0.15)), expr
# )
output = recursive_evaluate(expr.tensor(), sketcher)
fig = draw_contour_plot(output, 512, add_colorbar=False, levelrate=25)
fig.show()


### Resolved PolyArc CSG

Each translated copy is processed and merged correctly:


In [ ]:

poly_expr = expr_to_valid_polyset_expr(expr.tensor(), sketcher)
output = recursive_evaluate(poly_expr.tensor(), sketcher)
fig = draw_contour_plot(output, 512, add_colorbar=False, levelrate=25)
fig.show()



---

## Example 4: Simple Rectangle Difference

A simple case showing a rectangle with a translated rectangular hole. This demonstrates that even simple cases benefit from exact SDF evaluation:


In [ ]:

expr = gls.Difference(
    gls.Rectangle2D((1, 1,)),
    gls.Translate2D(gls.Rectangle2D((0.5, 0.9,)), (0.0, 0.3)),
)
output = recursive_evaluate(expr.tensor(), sketcher)
fig = draw_contour_plot(output, 512, add_colorbar=False, levelrate=25, solid_inside=False)
fig.show()

### Resolved PolyArc CSG

The resolved expression produces the same visual output, but with **exact distance values** throughout:


In [ ]:

poly_expr = expr_to_valid_polyset_expr(expr.tensor(), sketcher)
output = recursive_evaluate(poly_expr.tensor(), sketcher)
fig = draw_contour_plot(output, 512, add_colorbar=False, levelrate=25)
fig.show()

